In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import odeint

np.random.seed(2612)

In [2]:
# Configurações de cores e estilos
colors = {
    "GD": "gold",
    "AGD": "orangered", 
    "MD": "deepskyblue",
    "AMD1": "slateblue",
    "AMD2": "magenta",
    "AMDalt": "sienna",
}

estilo_setas = {
    'width': 0.003,
    'headwidth': 2,
    'headlength': 3
}

estilo_segmentos_aux = {
    'linestyle': 'dashed',
    'width': 0.003
}

In [3]:
shift = np.array([128., -56.])

def f(x):
    return (x[0] - shift[0])**2 + (x[1] - shift[1])**2 + (x[0] - shift[0])*(x[1] - shift[1])

def grad_f(x):
    return np.array([2*(x[0] - shift[0]) + x[1] - shift[1], 2*(x[1] - shift[1]) + x[0] - shift[0]])

a = 1
b = 1
c = 1.5

def phi(x):
    return a * x[0]**2 + b * x[1]**2 + c * x[0] * x[1]

def grad_phi(x):
    return np.array([2 * a * x[0] + c * x[1], 2 * b * x[1] + c * x[0]])

def inv_grad_phi(y):
    denom = 4 * a * b - c**2
    return np.array([(2 * b * y[0] - c * y[1])/denom, (2 * a * y[1] - c * y[0])/denom])

In [4]:
# Parâmetros fixos
T = 15  # Alterado para T=10
xopt = np.array([0., 0.]) + shift
theta0 = np.array([0., 0.])
x0 = inv_grad_phi(theta0)

# EDO para AMD
def AMD1_ode(state, t):
    theta1, theta2, vtheta1, vtheta2 = state
    dtheta1dt = vtheta1
    dtheta2dt = vtheta2
    x, y = inv_grad_phi([theta1, theta2])
    # Evitar divisão por zero
    if t < 1e-3:
        damping = 0
    else:
        damping = 3/t
    dvtheta1dt = - damping * vtheta1 - grad_f([x, y])[0]
    dvtheta2dt = - damping * vtheta2 - grad_f([x, y])[1]
    return [dtheta1dt, dtheta2dt, dvtheta1dt, dvtheta2dt]

# EDO para AMD suavizado
def AMDsuave_ode(state, t, delta):  # Mudando d para delta
    theta1, theta2, vtheta1, vtheta2 = state
    dtheta1dt = vtheta1
    dtheta2dt = vtheta2
    x, y = inv_grad_phi([theta1, theta2])
    # Usar max(t, delta) para suavização
    damping_time = max(t, delta)
    if damping_time < 1e-3:
        damping = 0
    else:
        damping = 3/damping_time
    dvtheta1dt = - damping * vtheta1 - grad_f([x, y])[0]
    dvtheta2dt = - damping * vtheta2 - grad_f([x, y])[1]
    return [dtheta1dt, dtheta2dt, dvtheta1dt, dvtheta2dt]

In [5]:
# Valores de delta para testar (alterados para 1, 2, 3, 4)
delta_values = [0.5, 1, 2, 4]

# PRIMEIRA PASSADA: Coletar todos os pontos para determinar os limites comuns
print("Coletando dados para determinar limites comuns...")
all_points = [x0, xopt]  # Incluir ponto inicial e ótimo

# Condição inicial para as EDOs - velocidade zero
initial_state_amd = [theta0[0], theta0[1], 0, 0]

# Coletar pontos das AMDsuave para todos os delta
t_ode_amd1 = np.linspace(0.1, T, 1000)

for delta in delta_values:
    sol_amdsuave = odeint(AMDsuave_ode, initial_state_amd, t_ode_amd1, args=(delta,))
    
    # Encontrar o ponto em t = delta
    idx_delta = np.argmin(np.abs(t_ode_amd1 - delta))
    theta_point_delta = sol_amdsuave[idx_delta, 0:2]
    x_point_delta = inv_grad_phi(theta_point_delta)
    all_points.append(x_point_delta)
    
    for i in range(len(sol_amdsuave)):
        theta_point = sol_amdsuave[i, 0:2]
        x_point = inv_grad_phi(theta_point)
        all_points.append(x_point)

# Coletar pontos da AMD1 (uma vez só, pois não depende de delta)
sol_amd1 = odeint(AMD1_ode, initial_state_amd, t_ode_amd1)
for i in range(len(sol_amd1)):
    theta_point = sol_amd1[i, 0:2]
    x_point = inv_grad_phi(theta_point)
    all_points.append(x_point)

# Converter para array e calcular limites comuns
all_points = np.array(all_points)
x_coords = all_points[:, 0]
y_coords = all_points[:, 1]

# Calcular limites comuns com padding
x_range_common = (x_coords.min() - 5, x_coords.max() + 5)
y_range_common = (y_coords.min() - 5, y_coords.max() + 5)

print(f"Limites comuns: x={x_range_common}, y={y_range_common}")

# SEGUNDA PASSADA: Gerar os gráficos
for delta in delta_values:
    print(f"Processando δ = {delta}")
    
    # Integrar as EDOs
    sol_amd1 = odeint(AMD1_ode, initial_state_amd, t_ode_amd1)
    sol_amdsuave = odeint(AMDsuave_ode, initial_state_amd, t_ode_amd1, args=(delta,))
    
    # Converter para espaço primal
    amd1_x = []
    amd1_y = []
    for i in range(len(sol_amd1)):
        theta_point = sol_amd1[i, 0:2]
        x_point = inv_grad_phi(theta_point)
        amd1_x.append(x_point[0])
        amd1_y.append(x_point[1])
    
    amdsuave_x = []
    amdsuave_y = []
    for i in range(len(sol_amdsuave)):
        theta_point = sol_amdsuave[i, 0:2]
        x_point = inv_grad_phi(theta_point)
        amdsuave_x.append(x_point[0])
        amdsuave_y.append(x_point[1])
    
    # Encontrar pontos em t = delta para ambas as trajetórias
    idx_delta = np.argmin(np.abs(t_ode_amd1 - delta))
    
    amd1_point_delta = [amd1_x[idx_delta], amd1_y[idx_delta]]
    amdsuave_point_delta = [amdsuave_x[idx_delta], amdsuave_y[idx_delta]]
    
    # Criar figura
    fig, ax = plt.subplots(figsize=(10, 10))
    
    # Grade para contornos (usando os limites comuns)
    xgrid = np.linspace(x_range_common[0], x_range_common[1], 100)
    ygrid = np.linspace(y_range_common[0], y_range_common[1], 100)
    X, Y = np.meshgrid(xgrid, ygrid)
    Z_f = f([X, Y])
    
    # Plot dos contornos
    ax.contour(X, Y, Z_f, levels=20, cmap='inferno', alpha=0.7)
    
    # Plot dos pontos importantes
    ax.scatter(x0[0], x0[1], color='k', s=100, label='ponto inicial', zorder=5)
    ax.scatter(xopt[0], xopt[1], color='gold', s=200, marker='*', 
               edgecolors='red', linewidth=1, label='ponto ótimo', zorder=5)
    
    # Plot das trajetórias
    amd1_plot, = ax.plot(amd1_x, amd1_y, color=colors['AMD1'], linewidth=2.5, 
                         label='AMD1 (δ=0)', zorder=3)
    
    amdsuave_plot, = ax.plot(amdsuave_x, amdsuave_y, color=colors['AMD2'], linewidth=2.5, 
                             label=f'AMDsuave (δ={delta})', zorder=3)
    
    # Plot dos pontos em t = delta (menos chamativos)
    amd1_delta_point = ax.scatter(amd1_point_delta[0], amd1_point_delta[1], 
                                 color=colors['AMD1'], s=60, marker='o', 
                                 edgecolors='black', linewidth=0.5, 
                                 label=f'AMD1 em t=δ', zorder=6)
    
    amdsuave_delta_point = ax.scatter(amdsuave_point_delta[0], amdsuave_point_delta[1], 
                                     color=colors['AMD2'], s=60, marker='s', 
                                     edgecolors='black', linewidth=0.5, 
                                     label=f'AMDsuave em t=δ', zorder=6)
    
    # Configurações do gráfico - USANDO LIMITES COMUNS
    ax.set_xlim(x_range_common)
    ax.set_ylim(y_range_common)
    ax.set_title(f'Comparação AMD1 vs AMDsuave (δ={delta}, T={T})')
    ax.set_xlabel('x')
    ax.set_ylabel('y')
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.3)
    
    # Legenda
    ax.legend(loc='upper right')
    
    # Salvar figura
    nome_base = f'AMD1_vs_AMDsuave_delta_{delta}_T_{T}'
    caminho_arquivo = f"imagens_dissertação/comparação_EDOS_suavizada_e_original/{nome_base}.pdf"
    fig.savefig(caminho_arquivo, bbox_inches="tight", dpi=300)
    plt.close(fig)  # Fechar figura para liberar memória
    
    print(f"Figura salva: {caminho_arquivo}")

print("Processamento concluído!")

Coletando dados para determinar limites comuns...
Limites comuns: x=(-5.0, 172.35938682491064), y=(-101.83081505342518, 5.0)
Processando δ = 0.5
Figura salva: imagens_dissertação/comparação_EDOS_suavizada_e_original/AMD1_vs_AMDsuave_delta_0.5_T_15.pdf
Processando δ = 1
Figura salva: imagens_dissertação/comparação_EDOS_suavizada_e_original/AMD1_vs_AMDsuave_delta_1_T_15.pdf
Processando δ = 2
Figura salva: imagens_dissertação/comparação_EDOS_suavizada_e_original/AMD1_vs_AMDsuave_delta_2_T_15.pdf
Processando δ = 4
Figura salva: imagens_dissertação/comparação_EDOS_suavizada_e_original/AMD1_vs_AMDsuave_delta_4_T_15.pdf
Processamento concluído!
